Importer les bibliothèques

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from IPython.display import Audio, display

from strauss.sonification import Sonification
from strauss.sources import Objects
from strauss.score import Score
from strauss.generator import Synthesizer

Charger les données 

In [ ]:
fichier = Path("../data/24_juillet_2025_donnees_preparees.csv")

df = pd.read_csv(fichier)

df["date"] = pd.to_datetime(df["Date/heure locale (UTC+2)"])

colonnes_pluie = list(df.columns[1:17])
stations = ["Q01", "Q02", "Q03", "Q04"]

print("Nombre de mesures :", len(df))
print("Nombre de pluviomètres :", len(colonnes_pluie))
print("Stations d'oxygène :", stations)

### Sonification pluie et oxygèene

Préparer les données de la pluie 
Conservation de deux indicateurs : l'intensité maximale et la proportion de stations où il pleut

In [ ]:
pluie = df[colonnes_pluie]

df["intensite_max"] = pluie.max(axis=1)
df["stations_actives"] = pluie.gt(0).sum(axis=1)
df["stations_valides"] = pluie.notna().sum(axis=1)

df["proportion_active"] = (
    df["stations_actives"] / df["stations_valides"]
)

intensite = df["intensite_max"].to_numpy(dtype=float)
etendue = df["proportion_active"].to_numpy(dtype=float)

reference_pluie = 10.0

hauteur = np.clip(intensite / reference_pluie, 0, 1)
volume = np.clip(etendue, 0, 1)

print("Hauteur maximale :", hauteur.max())
print("Volume maximal :", volume.max())

Préparer données oxygène
- référence commune de 10 mg/L pour les quatre stations

In [ ]:
reference_oxygene = 10.0

oxygene_normalise = {}

for station in stations:
    valeurs = df[station].to_numpy(dtype=float)

    oxygene_normalise[station] = np.clip(
        valeurs / reference_oxygene,
        0,
        1
    )

print("Données d'oxygène préparées")

Préparer le temps commun pour que pluie et oxygène soyent sincronisés 

In [ ]:
duree = 60

temps = np.linspace(0, 1, len(df))

print("Nombre d'instants :", len(temps))
print("Durée sonore :", duree, "secondes")

Sonifier la pluie

In [ ]:
score = Score([["A2"]], duree)

generator = Synthesizer()
generator.load_preset("pitch_mapper")

data_son = {
    "pitch": 1.0,
    "time_evo": temps,
    "pitch_shift": hauteur,
    "volume": volume,
    "azimuth": 0.5,
    "polar": 0.5
}

sources = Objects(data_son.keys())
sources.fromdict(data_son)
sources.apply_mapping_functions()

soni_pluie = Sonification(
    score,
    sources,
    generator,
    "stereo"
)

soni_pluie.render()

print("Sonification de pluie terminée")
soni_pluie.notebook_display()

Sonifier les quatre stations d'oxygène

In [ ]:
sons_oxygene = {}

positions = {
    "Q01": 0.25,
    "Q02": 0.42,
    "Q03": 0.58,
    "Q04": 0.75
}

for station in stations:

    data_son = {
        "pitch": 1.0,
        "time_evo": temps,
        "pitch_shift": oxygene_normalise[station],
        "volume": 0.35,
        "azimuth": positions[station],
        "polar": 0.5
    }

    sources = Objects(data_son.keys())
    sources.fromdict(data_son)
    sources.apply_mapping_functions()

    generator = Synthesizer()
    generator.load_preset("pitch_mapper")

    score = Score([["A2"]], duree)

    soni = Sonification(
        score,
        sources,
        generator,
        "stereo"
    )

    soni.render()

    sons_oxygene[station] = soni

    print("Sonification terminée :", station)

Écouter les quatre stations individuellement

In [ ]:
for station in stations:
    print("Station :", station)
    sons_oxygene[station].notebook_display()

Mélanger la pluie et l'oxygène

In [ ]:
import numpy as np
from IPython.display import Audio, display

# Fonction pour récupérer le véritable son STRAUSS
def extraire_audio(sonification):

    canaux = []
    frequence = None

    for stream in sonification.out_channels.values():

        # IMPORTANT : values et non samples
        audio = np.asarray(stream.values, dtype=float)
        canaux.append(audio)

        if frequence is None:
            frequence = int(stream.samprate)
        elif int(stream.samprate) != frequence:
            raise ValueError("Fréquences différentes")

    return np.stack(canaux, axis=0), frequence


# 1. Récupérer la pluie
piste_pluie, frequence = extraire_audio(soni_pluie)

# 2. Récupérer les quatre stations d'oxygène
pistes_oxygene = []

for station in stations:

    piste, freq = extraire_audio(sons_oxygene[station])

    if freq != frequence:
        raise ValueError(f"Fréquence différente : {station}")

    pistes_oxygene.append(piste)


# 3. Vérifier que les pistes sont compatibles
for piste in pistes_oxygene:
    if piste.shape != piste_pluie.shape:
        raise ValueError("Dimensions des pistes différentes")


# 4. Mélanger les quatre stations d'oxygène
mix_oxygene = np.mean(pistes_oxygene, axis=0)

# 5. Ajouter la pluie
mix_final = (
    0.5 * piste_pluie
    + 0.5 * mix_oxygene
)

# 6. Écouter les cinq sons ensemble
display(Audio(mix_final, rate=frequence, normalize=True))

Enregistrer le son pour l'associer ensuite à la carta

In [ ]:
import numpy as np
from scipy.io.wavfile import write
from pathlib import Path

# Récupérer les canaux audio de STRAUSS
canaux = []
frequence = None

for stream in soni_pluie.out_channels.values():
    canaux.append(np.asarray(stream.values, dtype=float))

    if frequence is None:
        frequence = int(stream.samprate)

# Transformer les canaux en tableau audio
audio_pluie = np.stack(canaux, axis=1)

# Normaliser le volume
maximum = np.max(np.abs(audio_pluie))

if maximum > 0:
    audio_pluie = audio_pluie / maximum

# Enregistrer le fichier WAV
dossier_audio = Path("../audio")
dossier_audio.mkdir(exist_ok=True)

fichier_wav = dossier_audio / "pluie_24_juillet_2025.wav"

write(
    fichier_wav,
    frequence,
    audio_pluie.astype(np.float32)
)

print("Fichier enregistré :", fichier_wav)
print("Durée :", len(audio_pluie) / frequence, "secondes")

Verifier que le son fonctionne

In [ ]:
from IPython.display import Audio, display

display(
    Audio(filename="../audio/pluie_24_juillet_2025.wav")
)

Le son ne foncitonne pas, verification du ficher wav:

In [ ]:
from pathlib import Path
from scipy.io import wavfile
import numpy as np

fichier = Path("../audio/pluie_24_juillet_2025.wav")

print("Fichier existe :", fichier.exists())

if fichier.exists():
    frequence, audio = wavfile.read(fichier)

    print("Fréquence :", frequence)
    print("Dimensions :", audio.shape)
    print("Durée :", len(audio) / frequence, "secondes")
    print("Amplitude maximale :", np.max(np.abs(audio)))

Tester directement le son de STRAUSS

In [ ]:
import numpy as np
from IPython.display import Audio, display

# Récupérer directement les canaux de STRAUSS
streams = list(soni_pluie.out_channels.values())

audio_test = np.stack(
    [np.asarray(s.values, dtype=float) for s in streams],
    axis=0
)

frequence_test = int(streams[0].samprate)

print("Dimensions :", audio_test.shape)
print("Fréquence :", frequence_test)
print("Durée :", audio_test.shape[1] / frequence_test)
print("Amplitude maximale :", np.max(np.abs(audio_test)))

display(
    Audio(
        data=audio_test,
        rate=frequence_test,
        normalize=True
    )
)